# Generate a nextnano Input File from a PHIDL Double Quantum Dot Layout

This notebook demonstrates the full translation pipeline for a **double quantum dot device**:

- build a parametric 2D gate layout using PHIDL-based device builders
- export the GDS/SVG layout
- extract the structured layout specification
- define the vertical process stack
- build a 3D simulation layout
- write a nextnano++ `.in` file from a reference template
- optionally run the simulation with `nextnanopy`
- inspect the generated structure and output folders

## 0. Setup

### 0.1 Imports

In [ ]:
import json
import os
import pprint
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from phidl import quickplot as qp

from qd_design import (
    LinearDotArrayDevice,
    build_simulation_layout,
    make_reference_sige_ge_process_stack,
    write_nextnano_input_from_template,
)
from nextnanopp_tools import (
    convergence_summary,
    get_bias_dir,
    get_output_directory,
    integrated_density_region_columns,
    list_variables,
    plot_bias_volume_3d,
    plot_bias_volume_slice,
    plot_convergence,
    plot_integrated_density_hole,
    plot_quantum_density_volume_3d,
    plot_quantum_density_volume_linecut,
    plot_quantum_density_volume_slice,
    plot_quantum_probability_volume_3d,
    plot_quantum_probability_volume_linecut,
    plot_quantum_probability_volume_slice,
    plot_structure_plane,
    plot_total_charges,
    plot_vtr_linecut,
    read_index_table,
    read_integrated_density_hole,
    read_total_charges,
    resolve_bias_output_file,
    resolve_quantum_output_file,
    resolve_quantum_probability_state_file,
    resolve_structure_file,
    run_input_file,
    validate_run_directory,
)

START_PATH = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (START_PATH, *START_PATH.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not locate the repository root from {START_PATH}; "
        "expected a parent containing pyproject.toml and src/."
    )

### 0.2 Helper Functions

In [ ]:
def print_json(data):
    print(json.dumps(data, indent=2))


def print_header(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

### 0.3 Configs

In [ ]:
# Repository paths are anchored independently of the Jupyter launch directory.
OUTPUT_DIR = REPO_ROOT / "data" / "gds"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENERATED_INPUT_DIR = REPO_ROOT / "configs" / "robert_inputs" / "generated"
GENERATED_INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reference nextnano input file used as template.
# Adjust this path if your template lives elsewhere.
TEMPLATE_INPUT_PATH = (
    REPO_ROOT
    / "configs"
    / "robert_inputs"
    / "double_qd"
    / "3d"
    / "Double_Quantum_Dot_3D.in"
)

# Generated files.
GDS_PATH = OUTPUT_DIR / "double_dot_from_phidl.gds"
SVG_PATH = OUTPUT_DIR / "double_dot_from_phidl.svg"
LAYOUT_SPEC_PATH = OUTPUT_DIR / "double_dot_from_phidl_layout_spec.json"
SIMULATION_LAYOUT_PATH = OUTPUT_DIR / "double_dot_from_phidl_simulation_layout.json"
GENERATED_INPUT_PATH = GENERATED_INPUT_DIR / "Double_Quantum_Dot_3D_from_PHIDL.in"

print("Template exists:", TEMPLATE_INPUT_PATH.exists())
print("Template path:", TEMPLATE_INPUT_PATH.resolve())
print("Generated input path:", GENERATED_INPUT_PATH.resolve())

## 1. Build the Double Quantum Dot Layout

### 1.1 Instantiate the Parametric Builder

In [ ]:
double_dot = LinearDotArrayDevice(
    name="double_dot_from_phidl",
    n_dots=2,

    device_y_size_nm=200.0,

    ohmic_width_nm=40.0,
    ohmic_length_nm=200.0,

    barrier_width_nm=40.0,
    barrier_length_nm=140.0,

    plunger_body_width_nm=40.0,
    plunger_body_length_nm=50.0,
    plunger_head_top_width_nm=60.0,
    plunger_head_max_width_nm=100.0,
    plunger_head_height_nm=100.0,
    plunger_upper_taper_height_nm=25.0,
    plunger_lower_taper_height_nm=25.0,

    ohmic_to_barrier_gap_nm=20.0,
    barrier_to_plunger_gap_nm=20.0,
)

layout = double_dot.ensure_built()

print_header("Double Dot Device Summary")
print_json(double_dot.summary())

### 1.2 Visualize the Layout

In [ ]:
qp(layout)

### 1.3 Export GDS and SVG

In [ ]:
double_dot.write_gds(str(GDS_PATH))
double_dot.write_svg(str(SVG_PATH))

print(f"Saved GDS: {GDS_PATH.resolve()}")
print(f"Saved SVG: {SVG_PATH.resolve()}")

## 2. Extract the Layout Spec

### 2.1 Inspect the Structured Layout Spec

In [ ]:
layout_spec = double_dot.layout_spec()

print_header("Layout Spec")
for element in layout_spec:
    print(
        element["name"],
        element["gate_type"],
        element["layer_name"],
        f"x=[{element['x_min_nm']}, {element['x_max_nm']}]",
        f"y=[{element['y_min_nm']}, {element['y_max_nm']}]",
        f"num_polygons={len(element.get('polygon_xy_nm', []))}",
    )

### 2.2 Save the Layout Spec

In [ ]:
with open(LAYOUT_SPEC_PATH, "w", encoding="utf-8") as f:
    json.dump(layout_spec, f, indent=2)

print(f"Saved layout spec: {LAYOUT_SPEC_PATH.resolve()}")

## 3. Define the Process Stack

### 3.1 Create the Reference SiGe/Ge Process Stack

In [ ]:
process_stack = make_reference_sige_ge_process_stack()

print_header("Process Stack")
print_json(process_stack.to_dict())

## 4. Build the 3D Simulation Layout

### 4.1 Combine Layout Spec and Process Stack

In [ ]:
simulation_layout = build_simulation_layout(
    name="double_dot_simulation_layout",
    layout_elements=layout_spec,
    process_stack=process_stack,
    x_margin_nm=0.0,
    y_margin_nm=0.0,
)

print_header("Simulation Layout Domain")
print_json(simulation_layout.domain.to_dict())

### 4.2 Inspect Background and Patterned Regions

In [ ]:
print_header("Background Regions")
for region in simulation_layout.background_regions:
    print(
        region.name,
        region.material,
        f"z=[{region.z_min_nm}, {region.z_max_nm}]",
        f"x=[{region.x_min_nm}, {region.x_max_nm}]",
        f"y=[{region.y_min_nm}, {region.y_max_nm}]",
    )

print_header("Patterned Regions")
for region in simulation_layout.patterned_regions:
    print(
        region.name,
        region.gate_type,
        region.layer_name,
        region.material,
        f"x=[{region.x_min_nm}, {region.x_max_nm}]",
        f"y=[{region.y_min_nm}, {region.y_max_nm}]",
        f"z=[{region.z_min_nm}, {region.z_max_nm}]",
        f"num_polygons={len(region.polygon_xy_nm)}",
    )

### 4.3 Save the Simulation Layout

In [ ]:
simulation_layout.write_json(str(SIMULATION_LAYOUT_PATH))

print(f"Saved simulation layout: {SIMULATION_LAYOUT_PATH.resolve()}")

## 5. Write the nextnano Input File

### 5.1 Generate the `.in` File from the Template

In [ ]:
if not TEMPLATE_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Template input file not found:\n{TEMPLATE_INPUT_PATH.resolve()}\n\n"
        "Copy your reference Double_Quantum_Dot_3D.in file to this path, "
        "or update TEMPLATE_INPUT_PATH in the config cell."
    )

write_nextnano_input_from_template(
    simulation_layout=simulation_layout,
    template_path=TEMPLATE_INPUT_PATH,
    output_path=GENERATED_INPUT_PATH,
    voltage_overrides={
        "V_P1": -3.0,
        "V_P2": -3.0,
        "V_B1": 0.0,
        "V_B2": 0.0,
        "V_B3": 0.0,
        "V_OC_L": 0.0,
        "V_OC_R": 0.0,
    },
)

print(f"Generated nextnano input file: {GENERATED_INPUT_PATH.resolve()}")

### 5.2 Inspect the Generated Input Snippet

In [ ]:
text = GENERATED_INPUT_PATH.read_text(encoding="utf-8")

print_header("First 120 Lines of Generated Input")
lines = text.splitlines()
for i, line in enumerate(lines[:120], start=1):
    print(f"{i:04d}: {line}")

### 5.3 Check That Polygonal Prisms Were Written

In [ ]:
n_polygonal_prisms = text.count("polygonal_prism")
n_region_blocks = text.count("region{")
n_contact_blocks = text.count("schottky{")

print("polygonal_prism count:", n_polygonal_prisms)
print("region{ count:", n_region_blocks)
print("schottky{ count:", n_contact_blocks)

In [ ]:
text = GENERATED_INPUT_PATH.read_text(encoding="utf-8")

checks = {
    "correct run quantum": "!WHEN $quantum            quantum{}" in text,
    "polygonal prisms": text.count("polygonal_prism"),
    "Body contact region": "contact{ name = Body }" in text,
    "remove_surface_charge region": "contact{ name = remove_surface_charge }" in text,
    "zero_fermi_QW region": "contact{ name = zero_fermi_QW }" in text,
    "top-level quantum blocks": text.count("\nquantum{"),
}

checks

## 6. Run the nextnano Simulation

### 6.1 Execute with a Tagged nextnano Output Folder

Local mode uses `run_input_file` to create a unique timestamped run under the external simulation-output root, then analyses the directory returned by the execution API. Transferred-run mode skips execution and analyses only the explicit completed `RUN_DIRECTORY` configured below.


In [ ]:
# Local mode: set RUN_SIMULATION=True and leave RUN_DIRECTORY=None.
# Transferred mode: keep RUN_SIMULATION=False and set RUN_DIRECTORY to a copied completed run.
RUN_SIMULATION = False
RUN_DIRECTORY: Path | None = None
RUN_TAG = "geometry_check_no_padding"
BIAS = "bias_00000"

SIMULATION_OUTPUT_ROOT = Path(
    os.environ.get(
        "NEXTNANO_OUTPUT_ROOT",
        REPO_ROOT.parent
        / "qpu-local-outputs"
        / "refactoring"
        / "runs",
    )
).expanduser().resolve()
SIMULATION_STAGING_ROOT = SIMULATION_OUTPUT_ROOT / "_staging"

if (
    SIMULATION_OUTPUT_ROOT == REPO_ROOT
    or REPO_ROOT in SIMULATION_OUTPUT_ROOT.parents
):
    raise ValueError("SIMULATION_OUTPUT_ROOT must be outside the repository.")

REQUIRED_OUTPUTS = (
    "Structure/materials.vtr",
    "potential.vtr",
    "bandedges.vtr",
    "density_hole.vtr",
    "Quantum/c-Ge_QW/HH/density.vtr",
    "iteration_quantum_poisson.dat",
    "integrated_density_hole.dat",
    "total_charges.txt",
)


In [ ]:
if RUN_SIMULATION:
    if RUN_DIRECTORY is not None:
        raise ValueError("Set RUN_DIRECTORY=None when RUN_SIMULATION=True.")

    if not GENERATED_INPUT_PATH.is_file():
        raise FileNotFoundError(
            f"Generated nextnano input file not found: {GENERATED_INPUT_PATH}"
        )

    executed_input = run_input_file(
        GENERATED_INPUT_PATH,
        output_root=SIMULATION_OUTPUT_ROOT,
        tag=RUN_TAG,
        add_timestamp=True,
        show_log=True,
        convergenceCheck=False,
        staging_root=SIMULATION_STAGING_ROOT,
        keep_staged_input=True,
    )

    RUN_DIRECTORY = get_output_directory(executed_input)
elif RUN_DIRECTORY is None:
    raise ValueError(
        "Set RUN_DIRECTORY to an explicit completed run when RUN_SIMULATION=False."
    )

RUN_DIRECTORY = validate_run_directory(
    RUN_DIRECTORY,
    bias=BIAS,
    require_complete=True,
    required_outputs=REQUIRED_OUTPUTS,
)
BIAS_DIRECTORY = get_bias_dir(RUN_DIRECTORY, BIAS)
STRUCTURE_DIRECTORY = RUN_DIRECTORY / "Structure"

print("Selected run directory:", RUN_DIRECTORY)
print("Selected bias directory:", BIAS_DIRECTORY)
print("Structure directory:", STRUCTURE_DIRECTORY)


### 6.2 Confirm the Explicit Run Selection

In [ ]:
print("Local execution enabled:", RUN_SIMULATION)
print("Configured local simulation output root:", SIMULATION_OUTPUT_ROOT)
print("Validated run directory:", RUN_DIRECTORY)

## 7. Simulated Structure and Run Diagnostics

The PHIDL layout and process-stack visualisations above describe the intended device geometry before simulation. The cells below inspect the structure actually written by nextnano in RUN_DIRECTORY and verify that the Ge quantum well, SiGe layers, oxide, and metallic contacts or gates are represented in the solver output.

### 7.1 Actual Simulated-Structure Summary and Mapping

The resolved files and lookup tables connect solver indices to material and contact names without hard-coding numerical indices.

In [ ]:
STRUCTURE_FILE_QUANTITIES = (
    "materials",
    "contacts",
    "regions_all",
)
STRUCTURE_FILES = {
    quantity: resolve_structure_file(
        RUN_DIRECTORY,
        quantity,
        preferred_extensions=("vtr",),
    )
    for quantity in STRUCTURE_FILE_QUANTITIES
}
STRUCTURE_FILE_SUMMARY = pd.DataFrame(
    [
        {"quantity": quantity, "filename": path.name, "path": path}
        for quantity, path in STRUCTURE_FILES.items()
    ]
)

MATERIAL_INDEX_PATH = resolve_structure_file(
    RUN_DIRECTORY,
    "material_indices",
    preferred_extensions=("txt",),
)
CONTACT_INDEX_PATH = resolve_structure_file(
    RUN_DIRECTORY,
    "contact_indices",
    preferred_extensions=("txt",),
)
MATERIAL_INDEX_TABLE = read_index_table(MATERIAL_INDEX_PATH)
CONTACT_INDEX_TABLE = read_index_table(CONTACT_INDEX_PATH)

display(STRUCTURE_FILE_SUMMARY)
display(MATERIAL_INDEX_TABLE)
display(CONTACT_INDEX_TABLE)

### 7.2 Representative Solver-Structure Views

The horizontal region plane uses the generated `xy_QW` section, while the vertical material plane uses the generated `xz_QW` section to verify the layer stack. The generated input does not define a compatible 1D structure section, so this workflow does not invent a structure line-cut coordinate.

In [ ]:
HORIZONTAL_STRUCTURE_FIGURE = plot_structure_plane(
    RUN_DIRECTORY,
    quantity="regions_all_2d_xy_QW",
    interactive=False,
)
VERTICAL_STRUCTURE_FIGURE = plot_structure_plane(
    RUN_DIRECTORY,
    quantity="materials_2d_xz_QW",
    interactive=False,
)

display(HORIZONTAL_STRUCTURE_FIGURE)
display(VERTICAL_STRUCTURE_FIGURE)

### 7.3 Quantum–Poisson Convergence

Summarise the final residuals for the explicitly selected bias and plot the iteration history without displaying the complete raw table.

In [ ]:
CONVERGENCE_SUMMARY = convergence_summary(BIAS_DIRECTORY)
display(CONVERGENCE_SUMMARY)

CONVERGENCE_FIGURE = plot_convergence(
    BIAS_DIRECTORY,
    interactive=False,
)
display(CONVERGENCE_FIGURE)

### 7.4 Integrated Hole Density

Read the run-level integrated density and show the final numerical values by solver region. Physical material labels require a compatible 1D structure DAT cut; this generated input defines only 2D structure sections, so the notebook does not guess that mapping.

In [ ]:
INTEGRATED_HOLE_DENSITY = read_integrated_density_hole(RUN_DIRECTORY)
INTEGRATED_REGION_COLUMNS = integrated_density_region_columns(
    INTEGRATED_HOLE_DENSITY
)

display(INTEGRATED_HOLE_DENSITY.loc[:, INTEGRATED_REGION_COLUMNS].tail(1))

INTEGRATED_HOLE_DENSITY_FIGURE = plot_integrated_density_hole(
    RUN_DIRECTORY,
    region_columns=INTEGRATED_REGION_COLUMNS,
    interactive=False,
    label_with_materials=False,
)
display(INTEGRATED_HOLE_DENSITY_FIGURE)

### 7.5 Total Charge

Read and display the selected bias's charge summary, then plot the reported charge quantities without saving the figure.

In [ ]:
TOTAL_CHARGES = read_total_charges(BIAS_DIRECTORY)
display(TOTAL_CHARGES)

TOTAL_CHARGES_FIGURE = plot_total_charges(
    BIAS_DIRECTORY,
    interactive=False,
)
display(TOTAL_CHARGES_FIGURE)

## 8. Inspect Physical Outputs

### 8.1 Hole Density Volume and Slices

Start with `density_hole.vtr`, the full 3D hole-density output in `bias_00000`. The first plot is a downsampled 3D Plotly volume focused around the quantum well. The second plot extracts an exact 2D plane from the same VTR file. Change `slice_axis` and `slice_value` to inspect arbitrary cuts.

In [ ]:
# DEFAULT_1D_LINE_DIAGNOSTIC: edit these values to move every default 1D line cut.
# For the planar double-dot layout, the default follows the dot axis through the QW mid-plane.
DEFAULT_1D_LINE_AXIS = "x"
DEFAULT_1D_LINE_FIXED_COORDS = {"y": 100.0, "z": -4.0}

DENSITY_HOLE_VTR = resolve_bias_output_file(
    RUN_DIRECTORY,
    "density_hole",
    bias=BIAS,
    preferred_extensions=("vtr",),
)

print("Physics run directory:", RUN_DIRECTORY)
print("Hole-density VTR:", DENSITY_HOLE_VTR.resolve())
print("Variables:", list_variables(DENSITY_HOLE_VTR))


#### 8.1.0 Default 1D Line-Cut Diagnostics <!-- DEFAULT_1D_LINE_DIAGNOSTIC -->

The default diagnostic follows the double-dot axis (`x`) while fixing `y = 100 nm` and `z = -7.5 nm` in the Ge quantum well. Change `DEFAULT_1D_LINE_AXIS` or `DEFAULT_1D_LINE_FIXED_COORDS` in the setup cell above to reuse every 1D diagnostic cell for another trace.


In [ ]:
# DEFAULT_1D_LINE_DIAGNOSTIC

def _format_fixed_coords_for_title(fixed_coords):
    return ", ".join(f"{name}={value:g} nm" for name, value in fixed_coords.items())


def plot_default_1d_diagnostic(
    quantity,
    variable,
    *,
    axis=None,
    fixed_coords=None,
    bias=None,
    yscale="linear",
    title=None,
):
    """Plot one editable 1D VTR diagnostic using the shared notebook defaults."""
    axis = DEFAULT_1D_LINE_AXIS if axis is None else axis
    fixed_coords = dict(DEFAULT_1D_LINE_FIXED_COORDS if fixed_coords is None else fixed_coords)
    bias = BIAS if bias is None else bias
    path = resolve_bias_output_file(
        RUN_DIRECTORY,
        quantity,
        bias=bias,
        preferred_extensions=("vtr",),
    )
    coord_label = _format_fixed_coords_for_title(fixed_coords)
    plot_title = title or f"{variable} along {axis} ({coord_label})"
    fig = plot_vtr_linecut(
        path,
        variable=variable,
        axis=axis,
        fixed_coords=fixed_coords,
        title=plot_title,
        interactive=True,
        yscale=yscale,
    )
    print(f"{quantity}/{variable}: {axis}-line with requested fixed coordinates {fixed_coords}")
    return fig


In [ ]:
# 3D overview of the hole density. The z range focuses on the quantum-well region;
# use coord_ranges=None to render a heavily downsampled view of the full simulated z domain.
hole_density_volume_fig = plot_bias_volume_3d(
    RUN_DIRECTORY,
    "density_hole",
    bias=BIAS,
    variable="Hole_density",
    title="Hole density near the quantum well",
    log10=False,
    coord_ranges={"z": (-30.0, 170.0)},
    max_points=150_000,
    mode="volume",       # use "isosurface" for sharper density contours
    opacity=0.18,
    surface_count=10,
)

In [ ]:
# Exact 2D plane extracted from density_hole.vtr.
# Fix z near the QW midpoint to get the lateral x-y hole-density profile.
hole_density_qw_slice_fig = plot_bias_volume_slice(
    RUN_DIRECTORY,
    "density_hole",
    bias=BIAS,
    variable="Hole_density",
    slice_axis="z",
    slice_value=-2.0,
    interactive=True,
    log10=False,
)

In [ ]:
# Other useful cuts through the same 3D VTR volume:

plot_bias_volume_slice(
    RUN_DIRECTORY, "density_hole", bias=BIAS, variable="Hole_density",
    slice_axis="x", slice_value=0.0, interactive=True, log10=False,
)

plot_bias_volume_slice(
    RUN_DIRECTORY, "density_hole", bias=BIAS, variable="Hole_density",
    slice_axis="y", slice_value=100.0, interactive=True, log10=False,
)

#### 8.1.1 Hole Density 1D Diagnostic <!-- DEFAULT_1D_LINE_DIAGNOSTIC -->

This line cut uses the same VTR volume as the 3D and 2D plots, but samples only the default double-dot-axis trace.


In [ ]:
# DEFAULT_1D_LINE_DIAGNOSTIC
hole_density_linecut_fig = plot_default_1d_diagnostic(
    "density_hole",
    "Hole_density",
    title="Hole density along the double-dot axis (y=100 nm, z=-7.5 nm)",
)


### 8.2 Electron Density Volume and Slices

Same workflow as the hole density, now using `density_electron.vtr`. The QW-plane slice fixes `z = -7.5 nm` to show the lateral electron-density profile.

In [ ]:
DENSITY_ELECTRON_VTR = resolve_bias_output_file(
    RUN_DIRECTORY,
    "density_electron",
    bias=BIAS,
    preferred_extensions=("vtr",),
)

print("Electron-density VTR:", DENSITY_ELECTRON_VTR.resolve())
print("Variables:", list_variables(DENSITY_ELECTRON_VTR))

In [ ]:
# 3D overview of the electron density near the quantum-well region.
electron_density_volume_fig = plot_bias_volume_3d(
    RUN_DIRECTORY,
    "density_electron",
    bias=BIAS,
    variable="Electron_density",
    title="Electron density near the quantum well",
    log10=False,
    coord_ranges={"z": (-30.0, 170.0)},
    max_points=150_000,
    mode="volume",
    opacity=0.18,
    surface_count=10,
)

In [ ]:
# Exact x-y QW-plane slice extracted from density_electron.vtr.
electron_density_qw_slice_fig = plot_bias_volume_slice(
    RUN_DIRECTORY,
    "density_electron",
    bias=BIAS,
    variable="Electron_density",
    slice_axis="z",
    slice_value=-7.5,
    interactive=True,
    log10=False,
)

#### 8.2.1 Electron Density 1D Diagnostic <!-- DEFAULT_1D_LINE_DIAGNOSTIC -->

The same trace is useful for checking whether the electron density remains negligible or develops a localized feature along the dot axis.


In [ ]:
# DEFAULT_1D_LINE_DIAGNOSTIC
electron_density_linecut_fig = plot_default_1d_diagnostic(
    "density_electron",
    "Electron_density",
    title="Electron density along the double-dot axis (y=100 nm, z=-7.5 nm)",
)


### 8.3 Electrostatic Potential Volume and Slices

Use `potential.vtr` for the same 3D view and exact QW-plane cut. Potential is signed, so this plot uses linear values rather than `log10`.

In [ ]:
POTENTIAL_VTR = resolve_bias_output_file(
    RUN_DIRECTORY,
    "potential",
    bias=BIAS,
    preferred_extensions=("vtr",),
)

print("Potential VTR:", POTENTIAL_VTR.resolve())
print("Variables:", list_variables(POTENTIAL_VTR))

In [ ]:
# 3D overview of the electrostatic potential near the quantum-well region.
potential_volume_fig = plot_bias_volume_3d(
    RUN_DIRECTORY,
    "potential",
    bias=BIAS,
    variable="Potential",
    title="Electrostatic potential near the quantum well",
    log10=False,
    coord_ranges={"z": (-30.0, 10.0)},
    max_points=150_000,
    mode="volume",
    opacity=0.18,
    surface_count=10,
)

In [ ]:
# Exact x-y QW-plane slice extracted from potential.vtr.
potential_qw_slice_fig = plot_bias_volume_slice(
    RUN_DIRECTORY,
    "potential",
    bias=BIAS,
    variable="Potential",
    # title="Electrostatic potential at the QW plane",
    slice_axis="z",
    slice_value=-2.0,
    interactive=True,
    log10=False,
)

#### 8.3.1 Potential 1D Diagnostic <!-- DEFAULT_1D_LINE_DIAGNOSTIC -->

This scan shows the electrostatic landscape along the default double-dot axis at the QW mid-plane.


In [ ]:
# DEFAULT_1D_LINE_DIAGNOSTIC
potential_linecut_fig = plot_default_1d_diagnostic(
    "potential",
    "Potential",
    title="Electrostatic potential along the double-dot axis (y=100 nm, z=-7.5 nm)",
)


### 8.4 Band Edges Volume and Slices

`bandedges.vtr` contains multiple variables and is much larger than the density/potential files. The cells below intentionally load one component at a time. Start with the HH band edge because it is usually the most relevant valence-band component for hole confinement, then run other components as needed.

In [ ]:
BANDEDGES_VTR = resolve_bias_output_file(
    RUN_DIRECTORY,
    "bandedges",
    bias=BIAS,
    preferred_extensions=("vtr",),
)

print("Band-edges VTR:", BANDEDGES_VTR.resolve())
print("Variables:", list_variables(BANDEDGES_VTR))

In [ ]:
# HH band edge: 3D overview near the quantum-well region.
hh_bandedges_volume_fig = plot_bias_volume_3d(
    RUN_DIRECTORY,
    "bandedges",
    bias=BIAS,
    variable="HH",
    title="HH band edge near the quantum well",
    log10=False,
    coord_ranges={"z": (-30.0, 10.0)},
    max_points=150_000,
    mode="volume",
    opacity=0.18,
    surface_count=10,
)

In [ ]:
# HH band edge: exact x-y QW-plane slice.
hh_bandedges_qw_slice_fig = plot_bias_volume_slice(
    RUN_DIRECTORY,
    "bandedges",
    bias=BIAS,
    variable="HH",
    title="HH band edge at the QW plane",
    slice_axis="z",
    slice_value=-4.0,
    interactive=True,
    log10=False,
)

In [ ]:
# LH band edge: exact x-y QW-plane slice.
lh_bandedges_qw_slice_fig = plot_bias_volume_slice(
    RUN_DIRECTORY,
    "bandedges",
    bias=BIAS,
    variable="LH",
    title="LH band edge at the QW plane",
    slice_axis="z",
    slice_value=-7.5,
    interactive=True,
    log10=False,
)

In [ ]:
# SO band edge: exact x-y QW-plane slice.
so_bandedges_qw_slice_fig = plot_bias_volume_slice(
    RUN_DIRECTORY,
    "bandedges",
    bias=BIAS,
    variable="SO",
    title="SO band edge at the QW plane",
    slice_axis="z",
    slice_value=-7.5,
    interactive=True,
    log10=False,
)

#### 8.4.1 Band-Edge 1D Diagnostics <!-- DEFAULT_1D_LINE_DIAGNOSTIC -->

Use the same line-cut geometry for the HH, LH, and SO band edges so their relative confinement profiles can be compared along the dot axis.


In [ ]:
# DEFAULT_1D_LINE_DIAGNOSTIC
bandedge_linecut_figs = {}
for bandedge_variable in ("HH", "LH", "SO"):
    bandedge_linecut_figs[bandedge_variable] = plot_default_1d_diagnostic(
        "bandedges",
        bandedge_variable,
        title=f"{bandedge_variable} band edge along the double-dot axis (y=100 nm, z=-7.5 nm)",
    )


In [ ]:
# Optional electron-relevant band-edge slices. Run these if you want conduction-band checks too.
#
# gamma_bandedges_qw_slice_fig = plot_bias_volume_slice(
#     RUN_DIRECTORY, "bandedges", bias=BIAS, variable="Gamma",
#     title="Gamma band edge at the QW plane", slice_axis="z", slice_value=-7.5,
#     interactive=True, log10=False,
# )
#
# delta1_bandedges_qw_slice_fig = plot_bias_volume_slice(
#     RUN_DIRECTORY, "bandedges", bias=BIAS, variable="Delta_1",
#     title="Delta_1 band edge at the QW plane", slice_axis="z", slice_value=-7.5,
#     interactive=True, log10=False,
# )
#
# l1_bandedges_qw_slice_fig = plot_bias_volume_slice(
#     RUN_DIRECTORY, "bandedges", bias=BIAS, variable="L_1",
#     title="L_1 band edge at the QW plane", slice_axis="z", slice_value=-7.5,
#     interactive=True, log10=False,
# )

In [ ]:
# DEFAULT_1D_LINE_DIAGNOSTIC
# Optional conduction-band 1D diagnostics. Uncomment the variables you want to inspect.
#
# for bandedge_variable in ("Gamma", "Delta_1", "L_1"):
#     bandedge_linecut_figs[bandedge_variable] = plot_default_1d_diagnostic(
#         "bandedges",
#         bandedge_variable,
#         title=f"{bandedge_variable} band edge along the double-dot axis (y=100 nm, z=-7.5 nm)",
#     )


## 9. Inspect Quantum Outputs

These cells inspect quantum-solver outputs once the run includes quantum calculations. The density plots use `density_<region>_<band>.vtr`; the shifted state-probability plots use the per-state `probability_shift_<region>_<band>_<state>.vtr` files.


### 9.1 Quantum Output Setup

Edit `QUANTUM_REGION`, `QUANTUM_BAND`, or `QUANTUM_STATE` to switch the quantum region, band, or eigenstate. The 1D quantum line cut reuses the same default axis and fixed coordinates as the physical-output diagnostics above.


In [ ]:
QUANTUM_REGION = "c-Ge_QW"
QUANTUM_BAND = "HH"
QUANTUM_KPOINT = "k00000"
QUANTUM_STATE = 1
QUANTUM_DENSITY_VARIABLE = "Density"
QUANTUM_PROBABILITY_VARIABLE = f"Psi^2_{QUANTUM_STATE}"
QUANTUM_QW_SLICE_Z_NM = DEFAULT_1D_LINE_FIXED_COORDS.get("z", -7.5)
QUANTUM_LINE_AXIS = DEFAULT_1D_LINE_AXIS
QUANTUM_LINE_FIXED_COORDS = dict(DEFAULT_1D_LINE_FIXED_COORDS)

QUANTUM_OUTPUTS_AVAILABLE = False
QUANTUM_MISSING_MESSAGE = ""
try:
    QUANTUM_DENSITY_VTR = resolve_quantum_output_file(
        RUN_DIRECTORY,
        "density",
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    QUANTUM_PROBABILITY_SHIFT_VTR = resolve_quantum_probability_state_file(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
        preferred_extensions=("vtr",),
    )
    QUANTUM_OUTPUTS_AVAILABLE = True
    print("Quantum density VTR:", QUANTUM_DENSITY_VTR.resolve())
    print("Quantum density variables:", list_variables(QUANTUM_DENSITY_VTR))
    print("Shifted probability VTR:", QUANTUM_PROBABILITY_SHIFT_VTR.resolve())
    print("Shifted probability variables:", list_variables(QUANTUM_PROBABILITY_SHIFT_VTR))
except FileNotFoundError as exc:
    QUANTUM_MISSING_MESSAGE = (
        "Quantum outputs were not found for this run. Re-run with quantum output enabled, "
        "or set RUN_DIRECTORY to a completed run that has bias_00000/Quantum outputs.\n"
        f"Details: {exc}"
    )
    print(QUANTUM_MISSING_MESSAGE)


### 9.2 Quantum-Calculated Hole Density

The next three cells show the quantum-calculated hole density in 3D, as an exact 2D QW-plane slice, and as the default 1D line cut.


In [ ]:
if QUANTUM_OUTPUTS_AVAILABLE:
    quantum_hole_density_volume_fig = plot_quantum_density_volume_3d(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        variable=QUANTUM_DENSITY_VARIABLE,
        title=f"Quantum-calculated {QUANTUM_BAND} hole density near the QW",
        log10=False,
        coord_ranges={"z": (-30.0, 10.0)},
        max_points=150_000,
        mode="volume",
        opacity=0.18,
        surface_count=10,
    )
else:
    print(QUANTUM_MISSING_MESSAGE)


In [ ]:
if QUANTUM_OUTPUTS_AVAILABLE:
    quantum_hole_density_qw_slice_fig = plot_quantum_density_volume_slice(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        variable=QUANTUM_DENSITY_VARIABLE,
        title=f"Quantum-calculated {QUANTUM_BAND} hole density at the QW plane",
        slice_axis="z",
        slice_value=QUANTUM_QW_SLICE_Z_NM,
        interactive=True,
        log10=False,
    )
else:
    print(QUANTUM_MISSING_MESSAGE)


In [ ]:
if QUANTUM_OUTPUTS_AVAILABLE:
    quantum_hole_density_linecut_fig = plot_quantum_density_volume_linecut(
        RUN_DIRECTORY,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        bias=BIAS,
        variable=QUANTUM_DENSITY_VARIABLE,
        axis=QUANTUM_LINE_AXIS,
        fixed_coords=QUANTUM_LINE_FIXED_COORDS,
        title=f"Quantum-calculated {QUANTUM_BAND} hole density along the default line cut",
        interactive=True,
    )
else:
    print(QUANTUM_MISSING_MESSAGE)


### 9.3 Shifted State Probability

These cells show the shifted probability density for `QUANTUM_STATE` as a 1D line cut, a 2D QW-plane slice, and a rotatable 3D volume.


In [ ]:
if QUANTUM_OUTPUTS_AVAILABLE:
    shifted_probability_linecut_fig = plot_quantum_probability_volume_linecut(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
        variable=QUANTUM_PROBABILITY_VARIABLE,
        axis=QUANTUM_LINE_AXIS,
        fixed_coords=QUANTUM_LINE_FIXED_COORDS,
        title=f"Shifted probability state {QUANTUM_STATE} along the default line cut",
        interactive=True,
    )
else:
    print(QUANTUM_MISSING_MESSAGE)


In [ ]:
if QUANTUM_OUTPUTS_AVAILABLE:
    shifted_probability_qw_slice_fig = plot_quantum_probability_volume_slice(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
        variable=QUANTUM_PROBABILITY_VARIABLE,
        title=f"Shifted probability state {QUANTUM_STATE} at the QW plane",
        slice_axis="z",
        slice_value=QUANTUM_QW_SLICE_Z_NM,
        interactive=True,
        log10=False,
    )
else:
    print(QUANTUM_MISSING_MESSAGE)


In [ ]:
if QUANTUM_OUTPUTS_AVAILABLE:
    shifted_probability_volume_fig = plot_quantum_probability_volume_3d(
        RUN_DIRECTORY,
        state=QUANTUM_STATE,
        region=QUANTUM_REGION,
        band=QUANTUM_BAND,
        kpoint=QUANTUM_KPOINT,
        bias=BIAS,
        variable=QUANTUM_PROBABILITY_VARIABLE,
        title=f"Shifted probability state {QUANTUM_STATE} near the QW",
        log10=False,
        coord_ranges={"z": (-30.0, 10.0)},
        max_points=150_000,
        mode="volume",
        opacity=0.18,
        surface_count=10,
    )
else:
    print(QUANTUM_MISSING_MESSAGE)
